In [1]:
from pathlib import Path
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Trỏ thẳng vào thư mục data bên trong project
BASE_DATA_DIR = PROJECT_ROOT / "data" / "Livestock_Dataset"

# --- CHỌN BỘ DỮ LIỆU ĐỂ TRAIN TẠI ĐÂY -

DATA_DIR = BASE_DATA_DIR / "Livestock_Skin"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print(f"Đang nạp dữ liệu từ: {DATA_DIR.name}")
print("Thư mục Data tồn tại:", DATA_DIR.exists())
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

TRAIN_MODE = True

Project root: C:\Users\MangOS\livestock-diseases-ai
Đang nạp dữ liệu từ: Livestock_Skin
Thư mục Data tồn tại: True
Torch: 2.5.1+cu121


Device: cuda


In [2]:
from src.utils import set_seed
from src.dataset import load_dataset, make_loaders
from src.model import build_model, count_parameters, unfreeze_backbone, load_checkpoint
from src.train import train_model
from src.evaluate import run_full_evaluation, plot_training_curves

set_seed(42)
print("Project modules imported successfully.")

Project modules imported successfully.


In [3]:
# Tự động đọc data và weights từ dataset.py mới
train_dataset, val_dataset, test_dataset, info = load_dataset(DATA_DIR)

train_loader, val_loader, test_loader = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32, # Giữ ở mức 32 để tránh tràn RAM
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

print("Dataset loaded successfully.")
print("Split sizes:", info["split_sizes"])
print("Number of classes:", info["n_classes"])
print("Trọng số phân lớp (Class Weights):", info["class_weights"])

Dataset loaded successfully.
Split sizes: {'train': 4892, 'val': 1047, 'test': 1060, 'total': 6999}
Number of classes: 10
Trọng số phân lớp (Class Weights): tensor([1.0004, 1.0004, 1.0004, 0.9923, 1.0004, 1.0004, 0.9984, 1.0128, 1.0004,
        0.9943])


In [4]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Min pixel value:", images.min().item())
print("Max pixel value:", images.max().item())
print("First 10 labels:", labels[:10].tolist())

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Min pixel value: -2.1179039478302
Max pixel value: 2.640000104904175
First 10 labels: [9, 7, 0, 8, 2, 4, 9, 4, 9, 0]


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tự động scale số lớp theo n_classes của dataset
model = build_model(
    n_classes=info["n_classes"],
    freeze_backbone=True,
).to(device)

params = count_parameters(model)

print("Device:", device)
print("Model device:", next(model.parameters()).device)
print("Model created successfully.")
print("Parameters:", params)

Device: cuda
Model device: cuda:0
Model created successfully.
Parameters: {'total': 11181642, 'trainable': 5130, 'frozen': 11176512}


In [6]:
with torch.no_grad():
    sample_outputs = model(images.to(device))

print("Sample output shape:", sample_outputs.shape)

Sample output shape: torch.Size([32, 10])


In [7]:
phase1_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=10,
    learning_rate=0.001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_pig_phase1_best.pth"),
    history_path=str(RESULTS_DIR / "history_pig_phase1.json"),
    phase_name="phase1_frozen_backbone",
)

print("Phase 1 training completed.")


Training phase1_frozen_backbone
Epochs: 10
Learning rate: 0.001
Trainable parameters: 5,130


train:   0%|                                           | 0/153 [00:00<?, ?it/s]

train:   1%|▏                                  | 1/153 [00:00<00:52,  2.89it/s]

train:   2%|▋                                  | 3/153 [00:00<00:19,  7.75it/s]

train:   3%|█▏                                 | 5/153 [00:00<00:13, 10.95it/s]

...

eval:  73%|██████████████████████████▏         | 24/33 [00:01<00:00, 19.86it/s]

eval:  82%|█████████████████████████████▍      | 27/33 [00:01<00:00, 20.01it/s]

eval:  91%|████████████████████████████████▋   | 30/33 [00:01<00:00, 20.09it/s]

eval: 100%|████████████████████████████████████| 33/33 [00:01<00:00, 20.87it/s]

Epoch 10/10 | Train Loss: 1.2967 | Train Acc: 0.5595 | Val Loss: 1.1332 | Val Acc: 0.6313 | Time: 10s
  --> Best checkpoint saved! (Val Acc: 0.6313)
Phase 1 training completed.


In [8]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_pig_phase1_best.pth"),
    device=device,
)

phase1_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase1",
    prefix="pig_",
)

plot_training_curves(
    phase1_history,
    save_path=FIGURES_DIR / "pig_training_curves_phase1.png",
)
print("Phase 1 evaluation and plotting completed.")

C:\Users\MangOS\livestock-diseases-ai\src\model.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


predict:   0%|                                          | 0/34 [00:00<?, ?it/s]

predict:   3%|█                                 | 1/34 [00:15<08:17, 15.08s/it]

predict:   9%|███                               | 3/34 [00:15<02:02,  3.95s/it]

predict:  18%|██████                            | 6/34 [00:15<00:43,  1.56s/it]

...

predict:  79%|██████████████████████████▏      | 27/34 [00:16<00:00,  8.87it/s]

predict:  88%|█████████████████████████████    | 30/34 [00:16<00:00, 10.74it/s]

predict:  97%|████████████████████████████████ | 33/34 [00:16<00:00, 12.57it/s]


--- Evaluation Results (phase1) ---
Accuracy   : 0.5972
Macro F1   : 0.5914
Weighted F1: 0.5916


Phase 1 evaluation and plotting completed.


In [9]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_pig_phase1_best.pth"),
    device=device,
)

unfreeze_backbone(model)

params = count_parameters(model)

print("Best Phase 1 checkpoint loaded.")
print("Backbone unfrozen for Phase 2.")
print("Parameters:", params)

Best Phase 1 checkpoint loaded.
Backbone unfrozen for Phase 2.
Parameters: {'total': 11181642, 'trainable': 11181642, 'frozen': 0}


C:\Users\MangOS\livestock-diseases-ai\src\model.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


In [10]:
phase2_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    learning_rate=0.0001, # LR nhỏ để tinh chỉnh mượt mà
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_pig_phase2_best.pth"),
    history_path=str(RESULTS_DIR / "history_pig_phase2.json"),
    phase_name="phase2_full_finetuning",
)

print("Phase 2 training completed.")


Training phase2_full_finetuning
Epochs: 15
Learning rate: 0.0001
Trainable parameters: 11,181,642


train:   0%|                                           | 0/153 [00:00<?, ?it/s]

train:   1%|▏                                  | 1/153 [00:00<01:40,  1.51it/s]

train:   1%|▍                                  | 2/153 [00:00<00:58,  2.60it/s]

train:   2%|▋                                  | 3/153 [00:01<00:42,  3.54it/s]

...

eval:  73%|██████████████████████████▏         | 24/33 [00:01<00:00, 19.94it/s]

eval:  82%|█████████████████████████████▍      | 27/33 [00:01<00:00, 20.27it/s]

eval:  91%|████████████████████████████████▋   | 30/33 [00:01<00:00, 20.10it/s]

eval: 100%|████████████████████████████████████| 33/33 [00:01<00:00, 21.19it/s]

Epoch 15/15 | Train Loss: 0.0747 | Train Acc: 0.9808 | Val Loss: 0.7137 | Val Acc: 0.8443 | Time: 26s
  --> Best checkpoint saved! (Val Acc: 0.8443)
Phase 2 training completed.


In [11]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_pig_phase2_best.pth"),
    device=device,
)

phase2_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase2",
    prefix="pig_",
)

plot_training_curves(
    phase2_history,
    save_path=FIGURES_DIR / "pig_training_curves_phase2.png",
)
print("Phase 2 evaluation completed.")

predict:   0%|                                          | 0/34 [00:00<?, ?it/s]

predict:   3%|█                                 | 1/34 [00:00<00:04,  7.02it/s]

predict:   9%|███                               | 3/34 [00:00<00:02, 13.44it/s]

predict:  18%|██████                            | 6/34 [00:00<00:01, 17.07it/s]

predict:  26%|█████████                         | 9/34 [00:00<00:01, 18.45it/s]

...

predict:  82%|███████████████████████████▏     | 28/34 [00:01<00:00, 20.32it/s]

predict:  91%|██████████████████████████████   | 31/34 [00:01<00:00, 20.25it/s]

predict: 100%|█████████████████████████████████| 34/34 [00:01<00:00, 22.55it/s]


--- Evaluation Results (phase2) ---
Accuracy   : 0.8132
Macro F1   : 0.8121
Weighted F1: 0.8122


Phase 2 evaluation completed.
